# Results File Audit — Load Everything, Confirm Classification

Pure diagnostic notebook, no plotting. Before trusting any ablation figure, this scans **every**
`*.json` under `RESULTS_DIR` (recursively — same as `load_matching_results` does) and answers:

1. **How many files are there, and where do they physically live** (which subfolder)?
2. **What task does each one actually belong to** (`mae_har` / `mae_hid` / unrecognized), based on
   its `exp` field — independent of which folder it's sitting in, since the HID-contamination bug
   showed folder location and task don't always agree.
3. **Within `mae_har`, which known experiment "section" does each file belong to** (A0 / A / B / C /
   D / E / F, matching the exact filter dicts from the visualization notebook) — and flags any file
   that matches **zero** sections (orphan / unrecognized config) or **more than one** (ambiguous).
4. **Is each file's eval data complete** (every checkpoint × layer × split has all the expected
   metrics), reusing the same completeness check as the visualization notebook.

Run this first, fix whatever it flags, *then* go back to the visualization notebook.


In [1]:

import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

# -- Paths ------------------------------------------------------------------
# Point this at the PARENT results directory (containing both mae_har/ and mae_hid/
# subfolders, or however your layout is organized) -- same directory the visualization
# notebook's RESULTS_DIR points at. This audit is most useful run against the parent,
# since that's exactly where cross-task contamination can happen.
RESULTS_DIR = Path("/home/zhuzih19/csi-project/csi-fall-detection/results")  # TODO: confirm

TEST_SPLITS = ["test_id", "test_cross_device", "test_cross_env", "test_cross_user"]
REQUIRED_METRICS = ["knn_acc", "lp_acc", "knn_f1", "lp_f1"]


## Step 1 — Load every JSON, record where it lives, catch parse errors

No filtering yet. Every file gets one row: path, parse status, `exp` name (if parseable), `args`
presence. Files that fail to parse or lack an `exp`/`args` key are flagged here rather than
silently disappearing from every later count.


In [2]:

all_json_paths = sorted(RESULTS_DIR.rglob("*.json"))
print(f"Found {len(all_json_paths)} .json files under {RESULTS_DIR} (recursive)")

file_records = []  # one dict per file, built up across the steps below

for fp in all_json_paths:
    rec = {
        "path": str(fp),
        "relative_path": str(fp.relative_to(RESULTS_DIR)),
        "parent_folder": fp.parent.name,
        "parse_ok": False,
        "exp": None,
        "has_args": False,
        "has_evals": False,
        "data": None,  # kept only in-memory for this run, not serialized into the DataFrame
    }
    try:
        text = fp.read_text()
        data = json.loads(text)
        rec["parse_ok"] = True
        rec["exp"] = data.get("exp")
        rec["has_args"] = data.get("args") is not None
        rec["has_evals"] = data.get("evals") is not None
        rec["data"] = data
    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        rec["parse_error"] = str(e)
    file_records.append(rec)

n_parse_fail = sum(1 for r in file_records if not r["parse_ok"])
n_no_exp = sum(1 for r in file_records if r["parse_ok"] and r["exp"] is None)
n_no_args = sum(1 for r in file_records if r["parse_ok"] and not r["has_args"])
n_no_evals = sum(1 for r in file_records if r["parse_ok"] and not r["has_evals"])

print(f"  parse failures:        {n_parse_fail}")
print(f"  parsed but no 'exp':   {n_no_exp}")
print(f"  parsed but no 'args':  {n_no_args}")
print(f"  parsed but no 'evals': {n_no_evals}")

if n_parse_fail:
    print("\nFiles that failed to parse:")
    for r in file_records:
        if not r["parse_ok"]:
            print(f"  {r['relative_path']}: {r['parse_error']}")


Found 321 .json files under /home/zhuzih19/csi-project/csi-fall-detection/results (recursive)
  parse failures:        0
  parsed but no 'exp':   216
  parsed but no 'args':  218
  parsed but no 'evals': 216


## Step 2 — Classify by task (`mae_har` / `mae_hid` / unrecognized)

Classification is by **`exp` name prefix** (matching the `task_prefix` guard now in
`load_matching_results`), not by folder — this is deliberately independent of physical location so
it catches exactly the kind of contamination we hit before (an HID file sitting somewhere a HAR
filter could still pick it up). Any mismatch between folder and task is flagged explicitly.


In [3]:

def classify_task(exp_name):
    if exp_name is None:
        return "unknown (no exp field)"
    if exp_name.startswith("mae_har"):
        return "mae_har"
    if exp_name.startswith("mae_hid"):
        return "mae_hid"
    return f"unrecognized prefix ({exp_name.split('_')[0] if exp_name else '?'})"


for r in file_records:
    r["task"] = classify_task(r["exp"]) if r["parse_ok"] else "parse_failed"

task_counts = defaultdict(int)
for r in file_records:
    task_counts[r["task"]] += 1

print("File counts by classified task:")
for task, n in sorted(task_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {task:<30} {n}")

# Folder-vs-task mismatch check: e.g. a file physically under a 'mae_hid' folder but whose
# exp name says 'mae_har', or vice versa. This is exactly how the earlier contamination bug
# could hide -- worth surfacing even if RESULTS_DIR is scoped correctly today.
print("\nFolder-vs-task mismatches (folder name doesn't contain the classified task string):")
mismatch_count = 0
for r in file_records:
    if r["task"] in ("mae_har", "mae_hid"):
        folder_lower = r["relative_path"].lower()
        if r["task"] not in folder_lower:
            print(f"  {r['relative_path']}: classified as '{r['task']}' but folder path doesn't say so")
            mismatch_count += 1
if mismatch_count == 0:
    print("  none")


File counts by classified task:
  unknown (no exp field)         216
  mae_har                        70
  mae_hid                        33
  unrecognized prefix (finetune) 2

Folder-vs-task mismatches (folder name doesn't contain the classified task string):
  none


## Step 3 — Within `mae_har`, classify by known experiment section

Filter dicts copied verbatim from the visualization notebook (`RATIO_ENC6_FILTERS`,
`RATIO_FILTERS`, `STRATEGY_FILTERS`, `STRATEGY_ENC12_FILTERS`, the square-patch check for `C`/`E`,
`HEAD_CAP_FILTERS`). A file can legitimately match **zero** sections (a one-off run, or a config
nobody's built a section for yet) or, if two filter dicts overlap, **more than one** — both cases
are printed explicitly rather than silently picking one.


In [4]:

SECTION_FILTERS = {
    "A0 (mask ratio, enc6)":      {"mask_strategy": "random", "encoder_depth": 6,  "patch_h": 29, "patch_w": 25},
    "A (mask ratio, enc12)":      {"mask_strategy": "random", "encoder_depth": 12, "patch_h": 29, "patch_w": 25},
    "B (mask strategy, enc6)":    {"mask_ratio": 0.75, "encoder_depth": 6,  "patch_h": 29, "patch_w": 25},
    "D (mask strategy, enc12)":   {"mask_ratio": 0.75, "encoder_depth": 12, "patch_h": 29, "patch_w": 25},
    "C (patch size, enc6)":       {"mask_ratio": 0.75, "encoder_depth": 6,  "mask_strategy": "random"},  # + square-patch check
    "E (patch size, enc12)":      {"mask_ratio": 0.75, "encoder_depth": 12, "mask_strategy": "random"},  # + square-patch check
    "F (head capacity)":          {"mask_strategy": "random", "encoder_depth": 12, "patch_h": 29, "patch_w": 25, "seed": 42},
}
SQUARE_PATCH_SECTIONS = {"C (patch size, enc6)", "E (patch size, enc12)"}


def matches_filter(args, filters):
    for key, expected in filters.items():
        actual = args.get(key)
        if actual is None:
            return False
        try:
            if isinstance(expected, float):
                if not abs(float(actual) - expected) < 1e-9:
                    return False
            elif isinstance(expected, int) and not isinstance(expected, bool):
                if int(actual) != expected:
                    return False
            else:
                if str(actual) != str(expected):
                    return False
        except (TypeError, ValueError):
            return False
    return True


har_records = [r for r in file_records if r["task"] == "mae_har" and r["has_args"]]

for r in har_records:
    args = r["data"]["args"]
    matched = []
    for section, filters in SECTION_FILTERS.items():
        if not matches_filter(args, filters):
            continue
        if section in SQUARE_PATCH_SECTIONS:
            ph, pw = args.get("patch_h"), args.get("patch_w")
            if ph is None or ph != pw:
                continue
        matched.append(section)
    r["matched_sections"] = matched

n_zero = sum(1 for r in har_records if len(r["matched_sections"]) == 0)
n_multi = sum(1 for r in har_records if len(r["matched_sections"]) > 1)
n_clean = sum(1 for r in har_records if len(r["matched_sections"]) == 1)

print(f"mae_har files: {len(har_records)} total")
print(f"  matched exactly 1 section: {n_clean}")
print(f"  matched 0 sections (orphan/unclassified): {n_zero}")
print(f"  matched >1 section (ambiguous overlap):    {n_multi}")

section_counts = defaultdict(int)
for r in har_records:
    for s in r["matched_sections"]:
        section_counts[s] += 1
print("\nPer-section file counts:")
for s in SECTION_FILTERS:
    print(f"  {s:<28} {section_counts.get(s, 0)}")

if n_zero:
    print("\nOrphan files (exp / key args, for you to eyeball):")
    for r in har_records:
        if len(r["matched_sections"]) == 0:
            a = r["data"]["args"]
            print(f"  {r['exp']}")
            print(f"    mask_ratio={a.get('mask_ratio')} mask_strategy={a.get('mask_strategy')} "
                  f"encoder_depth={a.get('encoder_depth')} patch_h={a.get('patch_h')} patch_w={a.get('patch_w')} "
                  f"seed={a.get('seed')}")

if n_multi:
    print("\nAmbiguous files (matched more than one section):")
    for r in har_records:
        if len(r["matched_sections"]) > 1:
            print(f"  {r['exp']}: {r['matched_sections']}")


mae_har files: 70 total
  matched exactly 1 section: 49
  matched 0 sections (orphan/unclassified): 12
  matched >1 section (ambiguous overlap):    9

Per-section file counts:
  A0 (mask ratio, enc6)        11
  A (mask ratio, enc12)        9
  B (mask strategy, enc6)      11
  D (mask strategy, enc12)     11
  C (patch size, enc6)         10
  E (patch size, enc12)        12
  F (head capacity)            5

Orphan files (exp / key args, for you to eyeball):
  mae_har_ep300_mask0.5_strategyrandom_enc6_dim128_bs128
    mask_ratio=0.5 mask_strategy=random encoder_depth=6 patch_h=None patch_w=None seed=None
  mae_har_ep300_mask0.625_strategyrandom_enc6_dim128_bs128
    mask_ratio=0.625 mask_strategy=random encoder_depth=6 patch_h=None patch_w=None seed=None
  mae_har_ep300_mask0.75_enc6_dim128_bs128
    mask_ratio=0.75 mask_strategy=time encoder_depth=6 patch_h=None patch_w=None seed=None
  mae_har_ep300_mask0.75_strategymixed_enc6_dim128_bs128
    mask_ratio=0.75 mask_strategy=mixed enc

## Step 4 — Eval-data completeness per file

Same completeness check as the visualization notebook's `diagnose_incomplete_runs`, run here
against every classified `mae_har` file up front, independent of which section it belongs to.


In [5]:

def get_layers(result):
    first_ckpt = next(iter(result["evals"].values()))
    layers = list(first_ckpt.keys())
    layers.sort(key=lambda x: int(re.search(r"\d+", x).group()))
    return layers


def get_checkpoints(result):
    ckpts = list(result["evals"].keys())
    ckpts.sort(key=lambda x: int(re.search(r"\d+", x).group()))
    return ckpts


def find_incomplete(result, required_metrics=REQUIRED_METRICS, splits=TEST_SPLITS):
    issues = []
    for ckpt in get_checkpoints(result):
        for layer in get_layers(result):
            for split in splits:
                entry = result["evals"].get(ckpt, {}).get(layer, {}).get(split)
                if entry is None:
                    issues.append(f"{ckpt}/{layer}: missing split '{split}' entirely")
                    continue
                missing = [m for m in required_metrics if m not in entry]
                if missing:
                    issues.append(f"{ckpt}/{layer}/{split}: missing metrics {missing}")
    return issues


n_complete = 0
n_incomplete = 0
for r in har_records:
    if not r["has_evals"]:
        r["completeness_issues"] = ["no 'evals' key at all"]
        n_incomplete += 1
        continue
    issues = find_incomplete(r["data"])
    r["completeness_issues"] = issues
    if issues:
        n_incomplete += 1
    else:
        n_complete += 1

print(f"mae_har files with fully complete eval data: {n_complete}")
print(f"mae_har files with at least one completeness issue: {n_incomplete}")

if n_incomplete:
    print("\nIncomplete files (first 5 issues shown per file):")
    for r in har_records:
        if r["completeness_issues"]:
            print(f"\n  {r['exp']}  ({r['relative_path']})")
            for issue in r["completeness_issues"][:5]:
                print(f"    - {issue}")
            if len(r["completeness_issues"]) > 5:
                print(f"    ... and {len(r['completeness_issues']) - 5} more")


mae_har files with fully complete eval data: 70
mae_har files with at least one completeness issue: 0


## Step 5 — Consolidated summary table

One row per `mae_har` file: task, matched section(s), completeness, seed — the single table to
scan before trusting any downstream ablation figure. Also flags duplicate `exp` names (two files
claiming to be the same run) since that silently causes `group_by` to average unrelated data
together if the loader wasn't careful about it.


In [6]:

summary_rows = []
for r in har_records:
    a = r["data"]["args"]
    summary_rows.append(dict(
        exp=r["exp"],
        relative_path=r["relative_path"],
        matched_sections=", ".join(r["matched_sections"]) if r["matched_sections"] else "(none)",
        n_sections_matched=len(r["matched_sections"]),
        complete="yes" if not r["completeness_issues"] else f"NO ({len(r['completeness_issues'])} issues)",
        mask_ratio=a.get("mask_ratio"), mask_strategy=a.get("mask_strategy"),
        encoder_depth=a.get("encoder_depth"), patch_h=a.get("patch_h"), patch_w=a.get("patch_w"),
        seed=a.get("seed"),
    ))

audit_df = pd.DataFrame(summary_rows)
with pd.option_context("display.max_rows", None, "display.width", 200):
    print(audit_df.to_string(index=False))

# Duplicate exp names -- two files claiming to be the exact same run silently get averaged
# together by group_by() as if they were two legitimate seeds of the same config.
dup_exps = audit_df["exp"].value_counts()
dup_exps = dup_exps[dup_exps > 1]
if len(dup_exps):
    print("\n[!] Duplicate 'exp' names found (same exp string, multiple files):")
    for exp_name, count in dup_exps.items():
        print(f"  {exp_name}: {count} files")
        for r in har_records:
            if r["exp"] == exp_name:
                print(f"    - {r['relative_path']}")
else:
    print("\nNo duplicate exp names among mae_har files.")

print(f"\n=== Bottom line ===")
print(f"Total files scanned:        {len(file_records)}")
print(f"mae_har (usable for viz):   {len(har_records)}")
print(f"  complete + classified:    {sum(1 for r in har_records if not r['completeness_issues'] and r['matched_sections'])}")
print(f"  needs attention:          {sum(1 for r in har_records if r['completeness_issues'] or not r['matched_sections'])}")
print(f"mae_hid (separate task):    {task_counts.get('mae_hid', 0)}")
print(f"other/unrecognized:         {sum(v for k, v in task_counts.items() if k not in ('mae_har', 'mae_hid'))}")


                                                                                    exp                                                                                        relative_path                                                   matched_sections  n_sections_matched complete  mask_ratio mask_strategy  encoder_depth  patch_h  patch_w  seed
                                 mae_har_ep300_mask0.5_strategyrandom_enc6_dim128_bs128                                  mae_har/mae_har_ep300_mask0.5_strategyrandom_enc6_dim128_bs128.json                                                             (none)                   0      yes       0.500        random              6      NaN      NaN   NaN
                mae_har_ep300_mask0.5_strategyrandom_ph29pw25_seed42_enc12_dim128_bs128                 mae_har/mae_har_ep300_mask0.5_strategyrandom_ph29pw25_seed42_enc12_dim128_bs128.json                           A (mask ratio, enc12), F (head capacity)                   2      yes       0.500    